# PlantSeg Round-2 -- Notebook I: SegFormer_b2 checkpoint reload, with the version pin Notebook C already found

**Why this notebook exists.** Notebook H (inference-only reload of all 6 Stage-2 checkpoints)
found a large gap for SegFormer_b2 only: fresh fg-IoU 0.2100 vs. the original/paper value
0.579/0.5792. Its own log showed why -- the checkpoint loaded **non-strictly**, with many
missing/unexpected keys, because Notebook H's setup cell imports `transformers` unpinned, and
whatever version Kaggle installs that day uses a different internal key layout than the one the
checkpoint was saved under months ago.

**This was already solved once.** Notebook C (investigating the same checkpoint for the
chance-baseline diagnostics) tried a version ladder and found `transformers==4.46.3` loads this
exact SegFormer_b2 checkpoint with **0 missing and 0 unexpected keys** -- a clean strict match.
Notebook C's own test there used the 300-image stratified subset, not the full 1561-image test
set, so it never checked whether the fixed load reproduces the full-test-set fg-IoU of 0.579.

**What this notebook does.** Pins `transformers==4.46.3`, reloads the SegFormer_b2 checkpoint,
confirms the load is strict (0/0 missing/unexpected, matching Notebook C), then re-evaluates on
the full 1561-image test set with the exact same mask-building, loader, and metric code Notebook
H used for all 6 models, so the resulting number is directly comparable to Notebook H's table and
to Table tab:seg. Only SegFormer_b2 is touched -- the other 5 models already matched (3 exactly,
2 within a small, likely hardware/nondeterminism gap) and are not rerun here.

## Setup -- pins transformers==4.46.3 before anything imports it

In [1]:
import importlib, os, subprocess, sys, time

def heartbeat(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

heartbeat("Notebook I starting: SegFormer_b2 reload with transformers==4.46.3 pinned")

for pkg, import_name in [("segmentation-models-pytorch", "segmentation_models_pytorch"),
                          ("albumentations", "albumentations")]:
    try:
        importlib.import_module(import_name)
        heartbeat(f"{import_name} already importable, skipping pip install")
    except ImportError:
        heartbeat(f"installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
        heartbeat(f"{pkg} installed")

# The pin that Notebook C already proved works for this exact checkpoint (0 missing / 0
# unexpected keys there). huggingface_hub must be pinned to a version contemporary with
# transformers==4.46.3 (pre-Xet) too -- otherwise Kaggle's current huggingface_hub/hf_xet
# combination breaks from_pretrained() with "ImportError: cannot import name
# 'XetDownloadProgressReporter' from 'huggingface_hub.utils._xet_progress_reporting'", a
# version-skew bug in huggingface_hub's newer Xet download backend unrelated to the
# checkpoint itself (found when Notebook I's first Kaggle run hit exactly this). Disable
# Xet outright, purge any already-imported transformers/huggingface_hub/hf_xet modules,
# and install both packages together so the resolved versions are self-consistent --
# matching Notebook C's try_version_ladder discipline.
os.environ["HF_HUB_DISABLE_XET"] = "1"
TRANSFORMERS_PIN = "4.46.3"
HF_HUB_PIN = "0.26.2"
for mod in list(sys.modules):
    if mod.split(".")[0] in ("transformers", "huggingface_hub", "hf_xet"):
        del sys.modules[mod]
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "hf_xet"])
heartbeat(f"installing huggingface_hub=={HF_HUB_PIN} transformers=={TRANSFORMERS_PIN} ...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                f"huggingface_hub=={HF_HUB_PIN}", f"transformers=={TRANSFORMERS_PIN}"],
               check=True)
heartbeat(f"huggingface_hub=={HF_HUB_PIN} / transformers=={TRANSFORMERS_PIN} installed")

heartbeat("imports starting")
import json, pickle, warnings
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd
from PIL import Image, ImageDraw
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import huggingface_hub as _hfh
from transformers import SegformerForSemanticSegmentation
import transformers as _tf
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
heartbeat(f"imports done, device={device}, transformers=={_tf.__version__}, "
          f"huggingface_hub=={_hfh.__version__}")
assert _tf.__version__ == TRANSFORMERS_PIN, (
    f"pinned transformers=={TRANSFORMERS_PIN} but {_tf.__version__} is what actually "
    f"imported -- stop and check the Kaggle environment before trusting the result below.")
assert _hfh.__version__ == HF_HUB_PIN, (
    f"pinned huggingface_hub=={HF_HUB_PIN} but {_hfh.__version__} is what actually "
    f"imported -- stop and check the Kaggle environment before trusting the result below.")

[15:46:11] Notebook I starting: SegFormer_b2 reload with transformers==4.46.3 pinned
[15:46:11] installing segmentation-models-pytorch ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 2.5 MB/s eta 0:00:00
[15:46:19] segmentation-models-pytorch installed
[15:46:30] albumentations already importable, skipping pip install
[15:46:31] installing huggingface_hub==0.26.2 transformers==4.46.3 ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 679.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 36.8 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.85.7 requires tokenizers<1.0,>=0.21.0, but you have tokenizers 0.20.3 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.26.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.26.2 which is incompatible.


[15:46:57] huggingface_hub==0.26.2 / transformers==4.46.3 installed
[15:46:57] imports starting


[15:47:40] imports done, device=cpu, transformers==4.46.3, huggingface_hub==0.26.2


## Config and checkpoint auto-discovery (SegFormer_b2 only)

In [2]:
SEG_TASK = "binary"; NUM_SEG = 2; IMG_SIZE = 512
OUTDIR = Path("/kaggle/working/s2_verify_notebookI"); MASK = OUTDIR/"masks"
for d in [OUTDIR, MASK]: d.mkdir(parents=True, exist_ok=True)

DATA_ROOT = None
for c in [Path("/kaggle/input/datasets/alenmaruf/plantseg/plantseg"), Path("/kaggle/input/plantseg"),
          Path("plantseg"), Path(".")]:
    if (c/"Metadata.csv").exists():
        DATA_ROOT = c; break
assert DATA_ROOT is not None, "Metadata.csv not found under any expected path -- attach the plantseg dataset."
heartbeat(f"DATA_ROOT = {DATA_ROOT}")

CKPT_NAME = "SegFormer_b2"
CKPT_PATH = None
for root in [Path("/kaggle/input"), Path(".")]:
    if not root.exists():
        continue
    for p in root.rglob("*.pt"):
        if p.stem == CKPT_NAME:
            CKPT_PATH = p; break
    if CKPT_PATH is not None:
        break
assert CKPT_PATH is not None, f"{CKPT_NAME}.pt not found -- attach the Stage-2 checkpoints dataset."
heartbeat(f"checkpoint found: {CKPT_NAME} -> {CKPT_PATH}")

[15:47:40] DATA_ROOT = /kaggle/input/datasets/alenmaruf/plantseg/plantseg
[15:48:27] checkpoint found: SegFormer_b2 -> /kaggle/input/datasets/alenmaruf/model-pt-class-seg-zip/model pt class+Seg/Segmentation/43seed_Seegmentation_checkpoints/SegFormer_b2.pt


## Index + binary masks, test split only (identical to Notebook H)

In [3]:
meta = pd.read_csv(DATA_ROOT/"Metadata.csv", encoding="utf-8-sig")
meta["split"] = meta["Split"].map({"Training": "train", "Validation": "val", "Test": "test"})
meta["filepath"] = meta.apply(lambda r: DATA_ROOT/"images"/r["split"]/r["Name"], axis=1)

def load_anns(split):
    d = json.load(open(DATA_ROOT/f"annotation_{split}.json")); by = defaultdict(list)
    for a in d["annotations"]:
        by[a["image_id"]].append(a)
    fn = {}
    for im in d["images"]:
        polys = [seg for a in by.get(im["id"], []) for seg in a["segmentation"] if len(seg) >= 6]
        fn[im["file_name"]] = (im["width"], im["height"], polys)
    return fn

def rasterize(w, h, polys):
    m = Image.new("L", (w, h), 0); dr = ImageDraw.Draw(m)
    for seg in polys:
        dr.polygon(list(zip(seg[0::2], seg[1::2])), fill=1)
    return m

heartbeat("building test-split binary masks ...")
fn = load_anns("test"); od = MASK/"test"; od.mkdir(parents=True, exist_ok=True)
recs = []
for r in meta[meta.split == "test"].itertuples():
    w, h, polys = fn.get(r.Name, (None, None, [])); mp = od/(Path(r.Name).stem + ".png")
    if not mp.exists():
        if w is None:
            with Image.open(r.filepath) as i0:
                w, h = i0.size
        rasterize(w, h, polys).save(mp)
    recs.append({"name": r.Name, "filepath": str(r.filepath), "mask": str(mp)})
test_df = pd.DataFrame(recs)
heartbeat(f"test set ready: {len(test_df)} images (expect 1561)")

[15:48:27] building test-split binary masks ...
[15:48:46] test set ready: 1561 images (expect 1561)


## Dataset / loader (identical to Notebook H)

In [4]:
MEAN = [0.485, 0.456, 0.406]; STD = [0.229, 0.224, 0.225]
eval_tf = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.Normalize(MEAN, STD), ToTensorV2()])

class SegDS(Dataset):
    def __init__(s, d, t): s.df = d.reset_index(drop=True); s.t = t
    def __len__(s): return len(s.df)
    def __getitem__(s, i):
        r = s.df.iloc[i]
        img = np.array(Image.open(r.filepath).convert("RGB")); m = np.array(Image.open(r["mask"]))
        a = s.t(image=img, mask=m)
        return a["image"], a["mask"].long()

def test_loader(bs):
    return DataLoader(SegDS(test_df, eval_tf), batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)

heartbeat("test loader ready")

[15:48:46] test loader ready


## Model (identical SegFormer wrapper Notebook H and the original training notebooks used)

In [5]:
class SegNet(nn.Module):
    def __init__(s, nc):
        super().__init__()
        s.net = SegformerForSemanticSegmentation.from_pretrained(
            "nvidia/mit-b2", num_labels=nc, ignore_mismatched_sizes=True)
    def forward(s, x):
        lg = s.net(pixel_values=x).logits
        return F.interpolate(lg, size=x.shape[-2:], mode="bilinear", align_corners=False)

heartbeat("model builder ready (encoder_weights not applicable here: HF SegFormer loads its own pretrained base, then we overwrite with the trained checkpoint below)")

[15:48:46] model builder ready (encoder_weights not applicable here: HF SegFormer loads its own pretrained base, then we overwrite with the trained checkpoint below)


## Metrics (identical Conf/evaluate code to Notebook H, so numbers are directly comparable)

In [6]:
class Conf:
    def __init__(s, n): s.n = n; s.cm = np.zeros((n, n), np.int64)
    def add(s, gt, pr):
        k = (gt >= 0) & (gt < s.n)
        s.cm += np.bincount(s.n * gt[k].astype(int) + pr[k], minlength=s.n ** 2).reshape(s.n, s.n)
    def scores(s):
        cm = s.cm.astype(float); tp = np.diag(cm)
        iou = tp / (cm.sum(1) + cm.sum(0) - tp + 1e-9); dice = 2 * tp / (cm.sum(1) + cm.sum(0) + 1e-9)
        pr = cm.sum(1) > 0
        return {"mIoU": float(np.nanmean(iou[pr])), "mDice": float(np.nanmean(dice[pr])),
                "pixel_acc": float(tp.sum() / (cm.sum() + 1e-9)),
                "fg_IoU": float(iou[1]), "fg_Dice": float(dice[1])}, iou

def evaluate(model, loader, n):
    model.eval(); cf = Conf(n)
    with torch.no_grad():
        for x, y in loader:
            pr = model(x.to(device)).argmax(1).cpu().numpy()
            cf.add(y.numpy().ravel(), pr.ravel())
    return cf

heartbeat("metrics ready")

[15:48:46] metrics ready


## Load checkpoint, confirm strict match, evaluate on the full test set

In [7]:
heartbeat(f"--- SegFormer_b2: loading checkpoint under transformers=={_tf.__version__} ---")
model = SegNet(NUM_SEG).to(device)
state = torch.load(CKPT_PATH, map_location=device)
missing_keys, unexpected_keys = model.load_state_dict(state, strict=False)
if missing_keys or unexpected_keys:
    heartbeat(f"  STILL non-strict: missing={missing_keys[:5]}... unexpected={unexpected_keys[:5]}... "
              f"-- the pin did not fix the key mismatch, do not trust the number below as a clean reload.")
    strict_ok = False
else:
    heartbeat("  state_dict loaded strictly, 0 missing/unexpected keys (matches Notebook C's finding)")
    strict_ok = True

heartbeat(f"--- SegFormer_b2: running full test-set inference ({len(test_df)} images) ---")
t0 = time.time()
cf = evaluate(model, test_loader(6), NUM_SEG)
sc, iou = cf.scores()
elapsed = time.time() - t0
heartbeat(f"--- SegFormer_b2: done in {elapsed:.1f}s | fresh test_scores = {{k: round(v,4) for k,v in sc.items()}} ---")

[15:48:46] --- SegFormer_b2: loading checkpoint under transformers==4.46.3 ---


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/99.0M [00:00<?, ?B/s]

Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/mit-b2 and are newly initialized: ['decode_head.batch_norm.bias', 'decode_head.batch_norm.num_batches_tracked', 'decode_head.batch_norm.running_mean', 'decode_head.batch_norm.running_var', 'decode_head.batch_norm.weight', 'decode_head.classifier.bias', 'decode_head.classifier.weight', 'decode_head.linear_c.0.proj.bias', 'decode_head.linear_c.0.proj.weight', 'decode_head.linear_c.1.proj.bias', 'decode_head.linear_c.1.proj.weight', 'decode_head.linear_c.2.proj.bias', 'decode_head.linear_c.2.proj.weight', 'decode_head.linear_c.3.proj.bias', 'decode_head.linear_c.3.proj.weight', 'decode_head.linear_fuse.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[15:48:50]   state_dict loaded strictly, 0 missing/unexpected keys (matches Notebook C's finding)
[15:48:50] --- SegFormer_b2: running full test-set inference (1561 images) ---
[16:39:52] --- SegFormer_b2: done in 3061.6s | fresh test_scores = {k: round(v,4) for k,v in sc.items()} ---


## Compare against Notebook H's fresh run, the preserved original pkl, and the paper table

In [8]:
ORIGINAL_PKL_SCORES = {"mIoU": 0.7226, "fg_IoU": 0.5792, "mDice": 0.8309, "pixel_acc": 0.8869}
PAPER_TABLE = {"fg_IoU": 0.579, "fg_Dice": 0.734, "mIoU": 0.723, "pixel_acc": 0.887}
NOTEBOOK_H_UNPINNED_FRESH = {"fg_IoU": 0.2100}  # Notebook H's result with transformers unpinned, for reference

fresh = sc["fg_IoU"]
orig = ORIGINAL_PKL_SCORES["fg_IoU"]
paper = PAPER_TABLE["fg_IoU"]
diff_orig = abs(fresh - orig)
status = "OK" if diff_orig < 0.01 else ("gap" if diff_orig < 0.05 else "LARGE GAP")

print(f"{'source':38s} {'fg-IoU':>8s}")
print(f"{'Notebook H (transformers unpinned)':38s} {NOTEBOOK_H_UNPINNED_FRESH['fg_IoU']:8.4f}")
print(f"{'Notebook I (transformers==4.46.3)':38s} {fresh:8.4f}")
print(f"{'preserved original result pkl':38s} {orig:8.4f}")
print(f"{'paper Table tab:seg':38s} {paper:8.4f}")
print()
print(f"strict load (0 missing/unexpected keys): {strict_ok}")
print(f"Notebook I vs. original pkl: diff={diff_orig:.4f} -> status={status}")
heartbeat("comparison printed above")

source                                   fg-IoU
Notebook H (transformers unpinned)       0.2100
Notebook I (transformers==4.46.3)        0.5792
preserved original result pkl            0.5792
paper Table tab:seg                      0.5790

strict load (0 missing/unexpected keys): True
Notebook I vs. original pkl: diff=0.0000 -> status=OK
[16:39:52] comparison printed above


## Save verification record

In [9]:
out = {"checkpoint_path": str(CKPT_PATH), "transformers_version_used": _tf.__version__,
       "strict_load": strict_ok, "fresh_test_scores": sc,
       "original_pkl_scores": ORIGINAL_PKL_SCORES, "paper_table": PAPER_TABLE,
       "notebook_h_unpinned_fresh": NOTEBOOK_H_UNPINNED_FRESH,
       "run_timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}
with open(OUTDIR/"notebookI_segformer_b2_version_pin_rerun.pkl", "wb") as f:
    pickle.dump(out, f)
heartbeat(f"SAVED {OUTDIR/'notebookI_segformer_b2_version_pin_rerun.pkl'}")
heartbeat("Notebook I complete. Bring this pkl (and the printed comparison above) back to the project folder.")

[16:39:52] SAVED /kaggle/working/s2_verify_notebookI/notebookI_segformer_b2_version_pin_rerun.pkl
[16:39:52] Notebook I complete. Bring this pkl (and the printed comparison above) back to the project folder.
